# 03 — Model Prototyping

Goals:
- Train a baseline autoencoder on *processed* features (128-dim)
- Verify training loop, shapes, checkpoints, and plots
- Compute reconstruction-loss anomaly scores

Prerequisite:
- Run notebook 02 to generate `data/processed/features_*.npz` first.

In [3]:
from __future__ import annotations

import sys
from pathlib import Path

# Allow importing `src.*` when running from the notebooks/ directory
REPO_ROOT = Path('..').resolve()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import torch
import matplotlib.pyplot as plt

from src.data.dataset import LHCDataset, build_dataloaders
from src.utils.config import get_model
from src.training.trainer import TrainConfig, train
from src.analysis.plotting import plot_loss_curves, plot_anomaly_scores

PROCESSED_DIR = REPO_ROOT / 'data' / 'processed'
OUT_DIR = REPO_ROOT / 'outputs' / 'models' / 'notebook_03'
FIG_DIR = REPO_ROOT / 'outputs' / 'figures' / 'notebook_03'
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

# Pick a processed feature file produced by notebook 02
FEATURE_FILE = PROCESSED_DIR / 'features_K30_events_LHCO2020_backgroundMC_Pythia.npz'
assert FEATURE_FILE.exists(), f"Missing {FEATURE_FILE}. Run notebook 02 first."

# Load dataset and dataloaders
dataset = LHCDataset(FEATURE_FILE)
train_loader, val_loader = build_dataloaders(dataset, batch_size=1024, seed=42)
input_dim = dataset.input_dim
print('Dataset:', FEATURE_FILE.name, 'Xdim=', input_dim, 'N=', len(dataset))

# Build model from config factory (autoencoder)
cfg = {'model': {'type': 'autoencoder', 'input_dim': input_dim, 'latent_dim': 16}}
model = get_model(cfg)
print('Model:', model.__class__.__name__)

# Train (quick baseline)
tc = TrainConfig(batch_size=1024, lr=1e-3, epochs=5, device='cpu', model_type='autoencoder')
trained_model, loss_log = train(model=model, train_loader=train_loader, val_loader=val_loader, config=tc, output_dir=OUT_DIR)

# Plot + save loss curves
train_losses = [e['train_loss'] for e in loss_log]
val_losses = [e['val_loss'] for e in loss_log]
plot_loss_curves(train_losses, val_losses, FIG_DIR / 'loss_curves.png')
print('Saved:', FIG_DIR / 'loss_curves.png')

# ---------------------------
# Anomaly scores (reconstruction MSE per event)
# ---------------------------
trained_model.eval()
device = torch.device('cpu')
scores = []
with torch.no_grad():
    for x, _y in val_loader:
        x = x.to(device)
        x_hat, _ = trained_model(x)
        mse = ((x_hat - x) ** 2).mean(dim=1)
        scores.append(mse.cpu().numpy())
scores = np.concatenate(scores)

plot_anomaly_scores(scores, labels=None, save_path=FIG_DIR / 'val_anomaly_scores.png')
print('Saved:', FIG_DIR / 'val_anomaly_scores.png')

plt.figure(figsize=(7, 4))
plt.hist(scores, bins=80, color='steelblue', alpha=0.8, density=True)
plt.title('Validation reconstruction-loss distribution')
plt.xlabel('MSE(x, x_hat)')
plt.ylabel('Density')
plt.grid(alpha=0.2)
plt.tight_layout()

Dataset: features_K30_events_LHCO2020_backgroundMC_Pythia.npz Xdim= 128 N= 200000
Model: SimpleAutoencoder
Epoch   1/5  |  train_loss: 0.626987  |  val_loss: 0.552954
Epoch   2/5  |  train_loss: 0.546846  |  val_loss: 0.542991
Epoch   3/5  |  train_loss: 0.540250  |  val_loss: 0.538885
Epoch   4/5  |  train_loss: 0.536706  |  val_loss: 0.535970
Epoch   5/5  |  train_loss: 0.533363  |  val_loss: 0.532218

Training complete. Best val_loss: 0.532218
Checkpoint saved to C:\Users\Pc\Desktop\LOCALS\DAMLALAB\ML_projects\LHC_olympics\outputs\models\notebook_03\best_model.pt
Saved: C:\Users\Pc\Desktop\LOCALS\DAMLALAB\ML_projects\LHC_olympics\outputs\figures\notebook_03\loss_curves.png
Saved: C:\Users\Pc\Desktop\LOCALS\DAMLALAB\ML_projects\LHC_olympics\outputs\figures\notebook_03\val_anomaly_scores.png
